<a href="https://colab.research.google.com/github/isaacmutuma/motion-analysis/blob/main/optical_flow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

# we upload our video
uploaded = files.upload() # the stored uploaded file as a dict
video_filename = list(uploaded.keys())[0] # convert all the keys(names)into a list and get the first position
file_size_mb = os.path.getsize(video_filename) / (1024 * 1024)

print(f"File loaded: {video_filename}")
print(f"File size: {file_size_mb:.2f} MB")

# we want to disect this video into frames using cap and extract info
cap = cv2.VideoCapture(video_filename) # a 'cursor' to read frame by frame

frame_count= int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps=cap.get(cv2.CAP_PROP_FPS)
width=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
duration=frame_count/ fps

print(f"Frames: {frame_count}")
print(f"FPS: {fps}")
print(f"Resolution: {width} x {height}")
print(f"Duration: {duration:.2f} seconds")

# we now read two frames and make sure ret is true
ret, frame1= cap.read() # two frames are stored as numpy arrays
ret, frame2=cap.read()# each array is a grid of pixels with RGB numbers

frame1_rgb = cv2.cvtColor(frame1, cv2.COLOR_BGR2RGB) # array converted to color
frame2_rgb = cv2.cvtColor(frame2, cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1, 2, figsize=(12, 4)) # 2 empty boxes on a canvas
axes[0].imshow(frame1_rgb) # we now add image 1
axes[0].set_title("Frame 1")
axes[0].axis("off")
axes[1].imshow(frame2_rgb)
axes[1].set_title("Frame 2")# we now add image 2
axes[1].axis("off")

plt.suptitle("Two consecutive frames — 33ms apart", fontsize=13)
plt.tight_layout()
plt.show()

# optical flow to detect motion between two frames
gray1 = cv2.cvtColor(frame1, cv2.COLOR_BGR2GRAY) # we use brightness to detect motion
gray2 = cv2.cvtColor(frame2, cv2.COLOR_BGR2GRAY) # # the brighteness of a moving arm stays detected
# the number of pixels and how they "flow" in both frames
flow = cv2.calcOpticalFlowFarneback( # the flow array ex (360, 640, 2) each with a motion vector
    gray1, gray2, # compare frames
    None,# no prior info
    pyr_scale=0.5, #shrink the frames for small and large movements
    levels=3,# levels start with small (blurry) and then zoom in
    winsize=15,# searches frame 2 to find the decsribed surface
    iterations=3,
    poly_n=5,# how we describe the brightness surface of a single pixel
    poly_sigma=1.2,# removes noise
    flags=0
)
print(f"Flow array shape: {flow.shape}")
print(f"Each pixel has: {flow.shape[2]} values (x-motion, y-motion)")

#cartToPolar we convert the raw values into how fast each pixel moved and which direction( pythgoras and arc tan)
magnitude, angle = cv2.cartToPolar(flow[..., 0], flow[..., 1]) # 2 grids of how far each pixel moved(from the motion vector)

#display the these numbers as images now angle(Hue-color),mag(speed-brightness)
hsv = np.zeros_like(frame1)# nothing yet
hsv[..., 0] =  angle * 180 / np.pi / 2 # direction as colors Hue
hsv[..., 1] = 255 # saturation is fully vivid
hsv[..., 2] = cv2.normalize(magnitude, None, 0, 255, cv2.NORM_MINMAX) # speed is brightness

flow_rgb = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR) # array convertions to color
flow_rgb = cv2.cvtColor(flow_rgb, cv2.COLOR_BGR2RGB) # reference of frame 1 converted to RGB

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].imshow(frame1_rgb)
axes[0].set_title("Original Frame")
axes[0].axis("off")
axes[1].imshow(flow_rgb)
axes[1].set_title("Optical Flow — direction=colour, speed=brightness")
axes[1].axis("off")

plt.tight_layout()
plt.show()






In [ ]:
# implement optical flow across all frames

# at this point cap is on frame 3 and we need it to set it to the start
cap.set(cv2.CAP_PROP_POS_FRAMES,0)
ret, prev_frame = cap.read() # reread frame 1
prev_grey= cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY) #convert it to gray scale
motion_over_time=[]
for i in range(1,frame_count):
  # we read the current frame
  ret,curr_frame=cap.read()
  if not ret:
    break
  curr_gray= cv2.cvtColor(curr_frame, cv2.COLOR_BGR2GRAY)
  # NOW WE HAVE TWO FRAMES FOR COMPARISON (PREV AND CURR)
  #CALCULATE THE FLOW ARRAY-how each pixel moved and its motion array
  flow = cv2.calcOpticalFlowFarneback(prev_grey, curr_gray, None, 0.5, 3, 15, 3, 5, 1.2, 0)
  # speed of motion
  magnitude, _ = cv2.cartToPolar(flow[..., 0], flow[..., 1]) # only need the speed this time
  mean_motion=magnitude.mean()
  motion_over_time.append(mean_motion) # the timeline of motion per pair


  prev_grey = curr_gray # Update prev_grey for the next iteration

print(f"Processed {len(motion_over_time)} frame pairs")
print(f"Max motion value: {max(motion_over_time):.4f}")
print(f"Mean motion value: {sum(motion_over_time)/len(motion_over_time):.4f}")


In [ ]:



# plotting motion over time

range_of_frame_numbers= range(len(motion_over_time))
#where in the video each frame pair it at
time_axis= [i/fps for i in range_of_frame_numbers]

plt.figure(figsize=(14, 5)) # canvas
plt.plot(time_axis, motion_over_time, color='steelblue', linewidth=1)
plt.axhline(y=np.mean(motion_over_time), color='red',
            linestyle='--', linewidth=1, label='Mean motion')

plt.xlabel("Time (seconds)")
plt.ylabel("Mean motion magnitude")
plt.title("Motion over time — optical flow magnitude per frame pair")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
cap.set(cv2.CAP_PROP_POS_FRAMES, 0)

output_path = "optical_flow_output.mp4" # where to save
fourcc = cv2.VideoWriter_fourcc(*'mp4v') # the video format
# output video file
out = cv2.VideoWriter(output_path, fourcc, fps, (width * 2, height)) # fps is similar to input file

ret, prev_frame = cap.read() # setting up the first frame
prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY) # setting the frame from colored to gray
# for all the frames
for i in range(1, frame_count):
    ret, curr_frame = cap.read()
    if not ret:
        break

    curr_gray = cv2.cvtColor(curr_frame, cv2.COLOR_BGR2GRAY)
#where we get our optical array
    flow = cv2.calcOpticalFlowFarneback(
        prev_gray, curr_gray,
        None, 0.5, 3, 15, 3, 5, 1.2, 0
    )
#we extract the motion vectors to make sense of the changes in magnitude and angle
    magnitude, angle = cv2.cartToPolar(flow[..., 0], flow[..., 1])
    #initialize our hsv to visualize the changes in real time
    hsv = np.zeros_like(curr_frame)
    hsv[..., 1] = 255 # saturation
    hsv[..., 0] = angle * 180 / np.pi / 2 # diffrent colours show diifrent angles
    hsv[..., 2] = cv2.normalize(magnitude, None, 0, 255, cv2.NORM_MINMAX) #brightness to show speed of change
    # array to color
    flow_bgr = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)

    combined = np.hstack((curr_frame, flow_bgr)) # we have the current orginal frame on the left and the flow bgr on the right
    out.write(combined) # show both

 # for the next iteration the prev gray becomes the current gray
    prev_gray = curr_gray

out.release()
print(f"Video saved to: {output_path}")
print(f"Resolution: {width * 2} x {height}")
print(f"Frames written: {frame_count - 1}")

In [20]:
import subprocess

subprocess.run([
    'ffmpeg', '-i', 'optical_flow_output.mp4',
    '-vf', 'scale=960:540',
    '-crf', '28',
    '-preset', 'fast',
    'optical_flow_compressed.mp4'
])

import os
size_mb = os.path.getsize('optical_flow_compressed.mp4') / (1024 * 1024)
print(f"Compressed size: {size_mb:.2f} MB")

print(os.path.getsize('optical_flow_compressed.mp4') / (1024 * 1024), "MB")



Compressed size: 2.17 MB
2.1667280197143555 MB


In [ ]:
files.download('optical_flow_compressed.mp4')